# 🥣 Ingredient Identification & Comprehensive Nutrition Version 2 Training### PyTorch Multi-Task Deep Neural Network (221 Food Categories)[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Incredient_Identification/NoteBooks/Ingredient%20Identification/Version2/ingredient_identification_v2_training.ipynb)---### 🚀 End-to-End Workflow:1. **Enable GPU (Optional but Recommended)**: `Runtime` -> `Change runtime type` -> **T4 GPU**.2. **Run All Cells**: Clones repository, loads 221-class dataset with 324 ingredient targets and 11 nutritional targets.3. **Dual-Head Training**: Trains embedding network for multi-label ingredients and normalized nutrition regression.4. **Analytical Plots**: Displays loss curves and validation metrics.5. **1-Click Download**: Automatically downloads `ingredient_identification_v2_best.pt` directly to your local computer.

## Step 1: Install Dependencies & Inspect HardwareInstalls PyTorch, Pandas, PyYAML, and Matplotlib.

In [ ]:
!pip install -q torch torchvision pandas numpy matplotlib pyyaml scikit-learn

import torch, sys
print('=' * 60)
print('🖥️  SYSTEM & HARDWARE SPECIFICATIONS')
print('=' * 60)
print(f'Python Version   : {sys.version.split()[0]}')
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {"✅ Yes" if torch.cuda.is_available() else "❌ No (CPU Mode)"}')
if torch.cuda.is_available():
    print(f'GPU Device Name  : {torch.cuda.get_device_name(0)}')
print('=' * 60)

## Step 2: Fresh GitHub Repository CloneClones the repository directly into `/content/` for fast access.

In [ ]:
import os

REPO_URL = 'https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git'
REPO_NAME = 'SouthIndian_Food_Nutrition'

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(REPO_NAME):
        print(f'🚀 Freshly cloning {REPO_NAME} from GitHub...')
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    print(f'✅ Active Working Directory: {os.getcwd()}')
else:
    print(f'Running in local workspace: {os.getcwd()}')

## Step 3: Load Version 2 Dataset (221 Food Categories)Loads `train.csv`, `val.csv`, `test.csv`, and `ingredients_vocabulary_v2.csv`.

In [ ]:
import pandas as pd, json

candidate_dirs = [
    'Incredient_Identification/DataSet/Version2 Used Data',
    os.path.join(os.getcwd(), 'Incredient_Identification', 'DataSet', 'Version2 Used Data'),
    '/content/SouthIndian_Food_Nutrition/Incredient_Identification/DataSet/Version2 Used Data'
]

DATASET_DIR = None
for d in candidate_dirs:
    if os.path.exists(d):
        DATASET_DIR = os.path.abspath(d)
        break

if not DATASET_DIR:
    raise FileNotFoundError('❌ Version 2 dataset directory not found!')

print(f'📁 Dataset Directory: {DATASET_DIR}')
train_df = pd.read_csv(os.path.join(DATASET_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(DATASET_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(DATASET_DIR, 'test.csv'))
vocab_df = pd.read_csv(os.path.join(DATASET_DIR, 'ingredients_vocabulary_v2.csv'))
classes_df = pd.read_csv(os.path.join(DATASET_DIR, 'food_and_nutrition_master_v2.csv'))

NUM_FOODS = len(classes_df)
NUM_INGREDIENTS = len(vocab_df)

print('=' * 60)
print(f'🔥 Total Food Classes      : {NUM_FOODS}')
print(f'🌿 Total Unique Ingredients: {NUM_INGREDIENTS}')
print(f'📊 Train Samples           : {len(train_df)}')
print(f'📊 Val Samples             : {len(val_df)}')
print(f'📊 Test Samples            : {len(test_df)}')
print('=' * 60)

## Step 4: Dataset Class & PyTorch DataLoaderConverts raw ingredients and multiple micronutrients into multi-hot and normalized regression tensors.

In [ ]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

ing2id = {r['ingredient_key'].lower(): int(r['ingredient_id']) for _, r in vocab_df.iterrows()}
id2ing = {int(r['ingredient_id']): r['display_name'] for _, r in vocab_df.iterrows()}
food2id = {r['food_name']: int(r['food_id']) for _, r in classes_df.iterrows()}
id2food = {int(r['food_id']): r['food_name'] for _, r in classes_df.iterrows()}

NUT_COLS = [
    'calories_kcal', 'protein_g', 'carbohydrates_g', 'fat_g', 'fiber_g',
    'sugar_g', 'sodium_mg', 'potassium_mg', 'calcium_mg', 'iron_mg', 'vitamin_c_mg'
]

class RecipeDataset(Dataset):
    def __init__(self, df, nut_mean=None, nut_std=None):
        self.food_ids = torch.tensor(df['food_id'].values, dtype=torch.long)
        
        labels = np.zeros((len(df), NUM_INGREDIENTS), dtype=np.float32)
        for idx, tokens in enumerate(df['ingredients_tokens']):
            for t in str(tokens).split(';'):
                t_clean = t.strip()
                if t_clean in ing2id:
                    labels[idx, ing2id[t_clean]] = 1.0
        self.ing_targets = torch.tensor(labels, dtype=torch.float32)
        
        nut_vals = df[NUT_COLS].values.astype(np.float32)
        if nut_mean is None:
            self.nut_mean = torch.tensor(nut_vals.mean(axis=0), dtype=torch.float32)
            self.nut_std = torch.tensor(nut_vals.std(axis=0) + 1e-6, dtype=torch.float32)
        else:
            self.nut_mean = nut_mean
            self.nut_std = nut_std
        self.nut_targets = (torch.tensor(nut_vals, dtype=torch.float32) - self.nut_mean) / self.nut_std
        
    def __len__(self):
        return len(self.food_ids)
        
    def __getitem__(self, idx):
        return self.food_ids[idx], self.ing_targets[idx], self.nut_targets[idx]

train_ds = RecipeDataset(train_df)
val_ds = RecipeDataset(val_df, train_ds.nut_mean, train_ds.nut_std)
test_ds = RecipeDataset(test_df, train_ds.nut_mean, train_ds.nut_std)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'✅ DataLoaders initialized: {len(train_loader)} train batches, {len(val_loader)} val batches.')

## Step 5: Define Multi-Task Neural Network ArchitectureDual-head neural network: Shared embedding trunk branching into ingredient multi-label classification and nutrition regression.

In [ ]:
import torch.nn as nn

class SouthIndianIngredientNutritionModel(nn.Module):
    def __init__(self, num_foods=221, num_ingredients=324, embed_dim=64, num_nutrition=11):
        super().__init__()
        self.food_embedding = nn.Embedding(num_foods, embed_dim)
        self.shared_trunk = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.ingredient_head = nn.Sequential(
            nn.Linear(128, 96),
            nn.ReLU(),
            nn.Linear(96, num_ingredients)
        )
        self.nutrition_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_nutrition)
        )

    def forward(self, food_ids):
        embed = self.food_embedding(food_ids)
        shared = self.shared_trunk(embed)
        return self.ingredient_head(shared), self.nutrition_head(shared)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SouthIndianIngredientNutritionModel(NUM_FOODS, NUM_INGREDIENTS, embed_dim=64, num_nutrition=len(NUT_COLS)).to(device)
print(model)

## Step 6: Train Multi-Task Model & Record MetricsTrains with joint BCEWithLogitsLoss (ingredients) and MSELoss (nutrition regression).

In [ ]:
import torch.optim as optim

EPOCHS = 50
LR = 0.002

bce_loss = nn.BCEWithLogitsLoss()
mse_loss = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

OUTPUT_DIR = 'Incredient_Identification/Models/Ingredient Identification/Version2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
BEST_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, 'best.pt')
LAST_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, 'last.pt')

print('=' * 65)
print('🚀 TRAINING INGREDIENT & NUTRITION PREDICTOR (VERSION 2)')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for f_ids, ing_tg, nut_tg in train_loader:
        f_ids, ing_tg, nut_tg = f_ids.to(device), ing_tg.to(device), nut_tg.to(device)
        optimizer.zero_grad()
        pred_ing, pred_nut = model(f_ids)
        loss = bce_loss(pred_ing, ing_tg) * 2.0 + mse_loss(pred_nut, nut_tg)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(f_ids)
    train_loss /= len(train_ds)
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for f_ids, ing_tg, nut_tg in val_loader:
            f_ids, ing_tg, nut_tg = f_ids.to(device), ing_tg.to(device), nut_tg.to(device)
            pred_ing, pred_nut = model(f_ids)
            loss = bce_loss(pred_ing, ing_tg) * 2.0 + mse_loss(pred_nut, nut_tg)
            val_loss += loss.item() * len(f_ids)
    val_loss /= len(val_ds)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'food2idx': food2id,
            'ing2idx': ing2id,
            'nut_mean': train_ds.nut_mean.tolist(),
            'nut_std': train_ds.nut_std.tolist(),
            'num_foods': NUM_FOODS,
            'num_ingredients': NUM_INGREDIENTS,
            'nut_cols': NUT_COLS,
            'epoch': epoch,
            'val_loss': best_val_loss
        }, BEST_WEIGHTS_PATH)
        
    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch [{epoch:2d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Best Val: {best_val_loss:.4f}')

print('\n✅ Training Finished!')
print(f'🏆 Best Weights Saved to: {BEST_WEIGHTS_PATH}')

## Step 7: Plot Training & Validation Loss CurvesVisualizes convergence progression across all training epochs.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Combined Loss Progression
ax1.plot(history['train_loss'], label='Train Combined Loss', color='#2ecc71', lw=2.5)
ax1.plot(history['val_loss'], label='Val Combined Loss', color='#e74c3c', lw=2.5, linestyle='--')
ax1.set_title('Dual-Head Training & Validation Loss', fontsize=14, pad=10)
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(fontsize=12)

# Validation Loss Zoom
epochs_range = range(1, len(history['val_loss']) + 1)
ax2.plot(epochs_range, history['val_loss'], label='Validation Convergence', color='#3498db', lw=2.5)
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1
best_loss = min(history['val_loss'])
ax2.scatter([best_epoch], [best_loss], color='#e67e22', s=120, zorder=5, label=f'Best Val: {best_loss:.4f} (Ep {best_epoch})')
ax2.set_title('Validation Loss Optimization (Peak Performance)', fontsize=14, pad=10)
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('Validation Loss', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(fontsize=12)

plt.tight_layout()
plt.savefig('ingredient_v2_loss_curve.png', dpi=200)
plt.show()
print(f'🏆 Best Validation Loss: {best_loss:.4f} achieved at Epoch {best_epoch}')

## Step 8: Test Inference on Diverse Food CategoriesDemonstrates predictions for South Indian dishes, non-veg, desserts, and fitness supplements.

In [ ]:
# Load best checkpoint
ckpt = torch.load(BEST_WEIGHTS_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

sample_dishes = ['Masala dosa', 'Chicken biryani', 'Whey protein', 'Avocado', 'Kheer']
nut_mean_t = torch.tensor(ckpt['nut_mean'], device=device)
nut_std_t = torch.tensor(ckpt['nut_std'], device=device)

print('=' * 65)
print('🍽️  INFERENCE VERIFICATION ON TEST DISHES')
print('=' * 65)

for dish in sample_dishes:
    if dish in food2id:
        fid = torch.tensor([food2id[dish]], device=device)
        with torch.no_grad():
            pred_ing, pred_nut = model(fid)
            probs = torch.sigmoid(pred_ing[0]).cpu().numpy()
            unnorm_nut = (pred_nut[0] * nut_std_t + nut_mean_t).cpu().numpy()
            
        top_indices = np.where(probs >= 0.25)[0]
        if len(top_indices) == 0:
            top_indices = np.argsort(probs)[-4:]
            
        pred_ingredients = [id2ing.get(idx, str(idx)) for idx in top_indices]
        print(f'\nFood: {dish}')
        print(f'  Predicted Ingredients: {", ".join(pred_ingredients)}')
        print(f'  Estimated Nutrition  : Calories: {unnorm_nut[0]:.1f} kcal, Protein: {unnorm_nut[1]:.1f}g, Carbs: {unnorm_nut[2]:.1f}g, Fat: {unnorm_nut[3]:.1f}g')

## Step 9: 📥 1-Click Automatic Model DownloadAutomatically downloads `ingredient_identification_v2_best.pt` directly to your local computer.

In [ ]:
# =====================================================================
# 💾 STEP 9: AUTOMATED OVERNIGHT PERSISTENCE & 1-CLICK AUTO-DOWNLOAD
# =====================================================================
# If training runs while you are away or overnight:
# 1. Permanently saves models to your personal Google Drive
# 2. Triggers automatic local browser download

import os, shutil

STANDARDIZED_NAME = 'ingredient_identification_v2_best.pt'
shutil.copy2(BEST_WEIGHTS_PATH, STANDARDIZED_NAME)

ING_FILES = [STANDARDIZED_NAME, 'ingredient_v2_loss_curve.png']

# --- 1. PERMANENT GOOGLE DRIVE BACKUP (SAFE OVERNIGHT STORAGE) ---
try:
    from google.colab import drive
    print('=' * 65)
    print('💾 SAVING TO GOOGLE DRIVE FOR PERMANENT OVERNIGHT SAFETY')
    print('=' * 65)
    drive.mount('/content/drive', force_remount=False)
    
    gdrive_dir = '/content/drive/MyDrive/Food_Inspection_V2_Models/Ingredient_Identification'
    os.makedirs(gdrive_dir, exist_ok=True)
    
    for f in ING_FILES:
        if os.path.exists(f):
            shutil.copy2(f, os.path.join(gdrive_dir, f))
            print(f'✅ Permanently backed up to Google Drive: {gdrive_dir}/{f}')
            
    print('\n🎉 SUCCESS: Ingredient Model permanently stored in your Google Drive!')
    print('Even if you close your browser or turn off your PC, your model is 100% safe.')
except Exception as e:
    print(f'Google Drive sync note: {e}')

# --- 2. AUTOMATIC BROWSER DOWNLOAD ---
try:
    from google.colab import files
    print('\n' + '=' * 65)
    print('📥 TRIGGERING AUTOMATIC DIRECT FILE DOWNLOAD')
    print('=' * 65)
    if os.path.exists(STANDARDIZED_NAME):
        print(f'Downloading {STANDARDIZED_NAME} to your computer...')
        files.download(STANDARDIZED_NAME)
except ImportError:
    print(f'Running locally: Model saved as {STANDARDIZED_NAME}')